# Óxidos: classificação e estequiometria
### Química — funções inorgânicas

**Objetivo:** Classificar óxidos e calcular a massa de CO₂ formada na decomposição do CaCO₃.

### Como usar este notebook
Execute as células em ordem. As de código conferem as contas e geram as visualizações. No 3D, **arraste com o mouse para girar em 360°** e use a roda para zoom.
A primeira célula instala somente dependências ausentes. O 3D precisa de acesso a `cdn.jsdelivr.net` no navegador e de um notebook confiável (trusted); se a rede bloquear o visualizador, o desenho 2D continua disponível.

Material para estudo teórico: não prove, toque nem misture produtos químicos para identificar uma função química.

In [ ]:
# Preparação idempotente: usa o Python do próprio kernel.
import importlib.util
import subprocess
import sys

for pacote, modulo in [("rdkit", "rdkit"), ("py3Dmol", "py3Dmol"),
                       ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=True)

import math
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display

## 1. Conceito principal

Óxidos são compostos binários em que o oxigênio é o elemento mais eletronegativo, usualmente com NOX −2. OF₂ não é óxido: é fluoreto de oxigênio. Peróxidos, como H₂O₂, têm oxigênio com NOX −1 e são tratados separadamente aqui.

| Classe | Exemplos e comportamento |
|---|---|
| Básicos | CaO: reage com ácido formando sal e água |
| Ácidos | CO₂: reage com base formando sal e água |
| Anfóteros | Al₂O₃: reage tanto com ácidos quanto com bases fortes |
| Neutros | CO e NO: não formam ácido/base por simples reação com água |

Nomes: CO₂ = dióxido de carbono; CaO = óxido de cálcio; Fe₂O₃ = óxido de ferro(III). Nem todo óxido metálico é básico e nem todo óxido não metálico é ácido.

$$\mathrm{CO_2+2NaOH\rightarrow Na_2CO_3+H_2O}$$
$$\boxed{n=\frac{m}{M},\qquad \frac{n(\mathrm{CO_2})}{n(\mathrm{CaCO_3})}=1}$$

## 2. Exemplo resolvido passo a passo

Na decomposição térmica completa de $10{,}0\ \mathrm{g}$ de CaCO₃ puro, qual massa de CO₂ se forma? Use massas molares arredondadas: 100,0 e 44,0 g/mol.

$$\mathrm{CaCO_3(s)\xrightarrow{\Delta}CaO(s)+CO_2(g)}$$
$$n(\mathrm{CaCO_3})=\frac{10{,}0}{100{,}0}=0{,}100\ \mathrm{mol}$$
$$n(\mathrm{CO_2})=0{,}100\ \mathrm{mol}$$
$$m(\mathrm{CO_2})=0{,}100\cdot44{,}0=\boxed{4{,}40\ \mathrm{g}}$$

O resultado supõe rendimento de 100% e não é um roteiro experimental.

## 3. Conferindo o cálculo com Python

In [ ]:
massa_inicial = 10.0  # g
n = massa_inicial / 100.0
massa_co2 = n * 44.0
print(f"CO2 = {n:.3f} mol; massa de CO2 = {massa_co2:.2f} g")
assert math.isclose(massa_co2, 4.4)

## 4. Visualização

### 4.1 Modelo 3D interativo

**Dióxido de carbono (CO₂).** C = cinza; O = vermelho. A molécula é linear (180°), com duas ligações duplas e comprimentos ilustrativos de 1,16 Å. O desenho 2D mostra O=C=O. CaO, citado na reação, é um sólido iônico e não deve ser imaginado como uma molécula semelhante ao CO₂.

In [ ]:
# Coordenadas didáticas em Å; não são uma simulação de solução.
SMILES = 'O=C=O'
COORDENADAS = [(-1.16, 0.0, 0.0), (0.0, 0.0, 0.0), (1.16, 0.0, 0.0)]
mol = Chem.MolFromSmiles(SMILES)
if mol is None:
    raise ValueError(f"SMILES inválido: {SMILES!r}")
mol = Chem.AddHs(mol)
if mol.GetNumAtoms() != len(COORDENADAS):
    raise ValueError("O número de coordenadas não corresponde ao de átomos.")
conf = Chem.Conformer(mol.GetNumAtoms())
for indice, posicao in enumerate(COORDENADAS):
    conf.SetAtomPosition(indice, posicao)
mol.AddConformer(conf)
visor = py3Dmol.view(width=640, height=400)
visor.addModel(Chem.MolToMolBlock(mol), "mol")
visor.setStyle({"stick": {"radius": 0.13, "colorscheme": "Jmol"},
                 "sphere": {"scale": 0.28, "colorscheme": "Jmol"}})
visor.setBackgroundColor("#ffffff")
visor.zoomTo()
visor.show()

### 4.2 Representação 2D

Cada traço representa uma ligação; dois traços representam uma ligação dupla.

In [ ]:
# Estrutura 2D: ligações covalentes com traços; íons separados por ponto
# no SMILES permanecem separados, sem inventar uma ligação covalente.
plano = Chem.Mol(mol)
rdDepictor.Compute2DCoords(plano)
desenho = rdMolDraw2D.MolDraw2DSVG(520, 300)
desenho.drawOptions().bondLineWidth = 2.2
desenho.DrawMolecule(plano)
desenho.FinishDrawing()
display(SVG(desenho.GetDrawingText()))

### 4.3 Gráfico de dados

In [ ]:
massas = [2.0 * k for k in range(11)]
plt.plot(massas, [m / 100.0 * 44.0 for m in massas], "o-")
plt.xlabel("Massa de CaCO₃ puro (g)")
plt.ylabel("Massa teórica de CO₂ (g)")
plt.title("Decomposição completa: proporção estequiométrica 1:1 em mol")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Como interpretar o resultado

Duas ligações duplas não significam quatro oxigênios: há dois átomos de O. A inclinação do gráfico vale 0,44 g de CO₂ por g de CaCO₃. O CaO retém a massa restante: 5,60 g no exemplo, conservando os 10,0 g totais em sistema fechado.

## 6. Exercício de fixação

Para 25,0 g de CaCO₃ puro, qual é a massa teórica de CO₂? Se o rendimento em CO₂ for 80%, qual massa é obtida?

Tente resolver antes de executar a resposta abaixo.

In [ ]:
teorica = 25.0 / 100.0 * 44.0
obtida = 0.80 * teorica
print(f"Massa teórica de CO2 = {teorica:.2f} g; obtida = {obtida:.2f} g")
assert math.isclose(teorica, 11.0) and math.isclose(obtida, 8.8)

## 7. Resumo

- Óxidos são compostos binários com oxigênio como elemento mais eletronegativo.
- Podem ser básicos, ácidos, anfóteros ou neutros.
- CO₂ é molecular e linear; CaO é iônico.
- Coeficientes da reação relacionam mol, não diretamente massas.

## Referências

- IUPAC. *Compendium of Chemical Terminology (Gold Book)*: https://goldbook.iupac.org/
- ATKINS, P.; JONES, L. *Princípios de Química*. Bookman.